<a href="https://colab.research.google.com/github/pritisanap2008-cyber/stackoverflow_tag_prediction/blob/main/stackoverflow_tag_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip install datasets -q
from datasets import load_dataset

stream = load_dataset("mikex86/stackoverflow-posts", split="train", streaming=True)

first = next(iter(stream))
print(list(first.keys()))
print({k: str(v)[:100] for k, v in first.items()})

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

['Id', 'PostTypeId', 'AcceptedAnswerId', 'ParentId', 'Score', 'ViewCount', 'Body', 'Title', 'ContentLicense', 'FavoriteCount', 'CreationDate', 'LastActivityDate', 'LastEditDate', 'LastEditorUserId', 'OwnerUserId', 'Tags']
{'Id': '7', 'PostTypeId': '2', 'AcceptedAnswerId': 'None', 'ParentId': '4', 'Score': '522', 'ViewCount': 'None', 'Body': "An explicit cast to `double` like this isn't necessary:\n\n```\ndouble trans = (double) trackBar1.Value", 'Title': 'None', 'ContentLicense': 'CC BY-SA 4.0', 'FavoriteCount': 'None', 'CreationDate': '2008-07-31T22:17:57.883', 'LastActivityDate': '2019-10-21T14:03:54.607', 'LastEditDate': '2019-10-21T14:03:54.607', 'LastEditorUserId': '5496973', 'OwnerUserId': '9', 'Tags': 'None'}


In [14]:
import pandas as pd
from datasets import load_dataset

stream = load_dataset("mikex86/stackoverflow-posts", split="train", streaming=True)

rows = []
for row in stream:
    if str(row["PostTypeId"]) == "1" and row["Tags"]:
        rows.append({"Title": row["Title"], "Body": row["Body"], "Tags": row["Tags"]})
    if len(rows) == 30000:
        break

df = pd.DataFrame(rows)
print(df.shape)
print(df[["Title", "Tags"]].head())
print(type(df["Tags"][0]))
print(df["Tags"][0])

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

(30000, 3)
                                               Title  \
0  How do I calculate someone's age based on a Da...   
1                      Calculate relative time in C#   
2  Difference between Math.Floor() and Math.Trunc...   
3  Why did the width collapse in the percentage w...   
4                        Determine a user's timezone   

                                                Tags  
0                               [c#, .net, datetime]  
1  [c#, datetime, time, datediff, relative-time-s...  
2                                       [.net, math]  
3                   [html, css, internet-explorer-7]  
4  [html, browser, timezone, user-agent, timezone...  
<class 'list'>
['c#', '.net', 'datetime']


In [15]:
from collections import Counter
import re

all_tags = Counter(tag for tags in df["Tags"] for tag in tags)
print(all_tags.most_common(10))

top_tags = [tag for tag, count in all_tags.most_common(50)]

df["Tags"] = df["Tags"].apply(lambda tags: [t for t in tags if t in top_tags])
df = df[df["Tags"].apply(len) > 0].reset_index(drop=True)

text = df["Title"].fillna("") + " " + df["Body"].fillna("")
text = text.str.replace(r"```.*?```", " ", regex=True, flags=re.DOTALL)
text = text.str.replace(r"<[^>]+>", " ", regex=True)
text = text.str.replace(r"http\S+", " ", regex=True)
text = text.str.lower()
text = text.str.replace(r"[^a-z0-9#+ ]", " ", regex=True)
text = text.str.replace(r"\s+", " ", regex=True).str.strip()
df["text"] = text

print(df.shape)
print(df["text"][0][:300])
print(df["Tags"][0])

[('c#', 3746), ('.net', 3269), ('java', 2305), ('asp.net', 1825), ('c++', 1599), ('javascript', 1389), ('sql-server', 1192), ('php', 1141), ('python', 1081), ('sql', 1079)]
(23432, 4)
how do i calculate someone s age based on a datetime type birthday given a datetime representing a person s birthday how do i calculate their age in years
['c#', '.net']


In [16]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

mlb = MultiLabelBinarizer(classes=top_tags)
y = mlb.fit_transform(df["Tags"])

text_train, text_test, y_train, y_test = train_test_split(
    df["text"], y, test_size=0.2, random_state=42)

tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2),
                        min_df=3, token_pattern=r"[a-z0-9#+]+")
X_train = tfidf.fit_transform(text_train)
X_test = tfidf.transform(text_test)

print(y.shape)
print(X_train.shape, X_test.shape)
print(list(mlb.classes_[:10]))

(23432, 50)
(18745, 20000) (4687, 20000)
['c#', '.net', 'java', 'asp.net', 'c++', 'javascript', 'sql-server', 'php', 'python', 'sql']


In [17]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score, hamming_loss

clf = OneVsRestClassifier(LogisticRegression(C=10, solver="liblinear"), n_jobs=-1)
clf.fit(X_train, y_train)

probs = clf.predict_proba(X_test)

pred_05 = (probs >= 0.5).astype(int)
pred_03 = (probs >= 0.3).astype(int)

print("Threshold 0.5")
print("  micro F1:", f1_score(y_test, pred_05, average="micro", zero_division=0))
print("  macro F1:", f1_score(y_test, pred_05, average="macro", zero_division=0))
print("  precision:", precision_score(y_test, pred_05, average="micro", zero_division=0))
print("  recall:", recall_score(y_test, pred_05, average="micro", zero_division=0))
print("  hamming loss:", hamming_loss(y_test, pred_05))

print("Threshold 0.3")
print("  micro F1:", f1_score(y_test, pred_03, average="micro", zero_division=0))
print("  macro F1:", f1_score(y_test, pred_03, average="macro", zero_division=0))
print("  precision:", precision_score(y_test, pred_03, average="micro", zero_division=0))
print("  recall:", recall_score(y_test, pred_03, average="micro", zero_division=0))
print("  hamming loss:", hamming_loss(y_test, pred_03))

i = 0
print(text_test.iloc[i][:200])
print("True:", [t for t, v in zip(mlb.classes_, y_test[i]) if v == 1])
print("Predicted:", [t for t, v in zip(mlb.classes_, pred_03[i]) if v == 1])

Threshold 0.5
  micro F1: 0.5889118519184113
  macro F1: 0.5573649531559911
  precision: 0.8137571393096598
  recall: 0.4614193185018305
  hamming loss: 0.019522082355451248
Threshold 0.3
  micro F1: 0.6458946029727367
  macro F1: 0.6208977772234213
  precision: 0.7415586785909838
  recall: 0.5720923683469445
  hamming loss: 0.01901002773629187
which db should i select if performance of postgres is low in a web app that support more than 5000 users postgres is becoming the bottle neck it takes more than 1 minute to add a new user even after 
True: ['database', 'performance']
Predicted: ['database', 'performance']
